
# AI-Based Image Enhancement System (Updated - Fixed VAE)

Includes:
- Denoising Autoencoder
- Fixed Variational Autoencoder (No KerasTensor Error)
- GAN (Basic)

Dataset: CIFAR-10


In [ ]:
pip install numpy tensorflow==2.16.1 matplotlib pillow

In [ ]:

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import backend as K


: 

In [3]:

(x_train, _), (x_test, _) = tf.keras.datasets.cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0


## Denoising Autoencoder

In [4]:

input_img = layers.Input(shape=(32,32,3))
x = layers.Conv2D(32, 3, activation='relu', padding='same')(input_img)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
x = layers.UpSampling2D(2)(x)
decoded = layers.Conv2D(3, 3, activation='sigmoid', padding='same')(x)

autoencoder = models.Model(input_img, decoded)
autoencoder.compile(optimizer='adam', loss='mse')


W0000 00:00:1776551278.948109    2850 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1776551279.448897    1273 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## Fixed Variational Autoencoder

In [5]:

latent_dim = 64

inputs = layers.Input(shape=(32,32,3))
x = layers.Flatten()(inputs)
x = layers.Dense(128, activation='relu')(x)

z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)

def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=(tf.shape(z_mean)[0], latent_dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = layers.Lambda(sampling)([z_mean, z_log_var])

decoder_input = layers.Input(shape=(latent_dim,))
x = layers.Dense(128, activation='relu')(decoder_input)
x = layers.Dense(32*32*3, activation='sigmoid')(x)
decoder_output = layers.Reshape((32,32,3))(x)

decoder = models.Model(decoder_input, decoder_output)
outputs = decoder(z)

vae = models.Model(inputs, outputs)

def vae_loss(y_true, y_pred):
    reconstruction_loss = K.mean(K.square(y_true - y_pred))
    kl_loss = -0.5 * K.mean(
        1 + z_log_var - K.square(z_mean) - K.exp(z_log_var)
    )
    return reconstruction_loss + kl_loss

vae.compile(optimizer='adam', loss=vae_loss)
vae.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 3072)      │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    393,344 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 64)        │          0 │ dense_1[0][0],    │
│                     │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_1        │ (None, 32, 32, 3) │    404,608 │ lambda[0][0]      │
│ (Functional)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 814,464 (3.11 MB)

 Trainable params: 814,464 (3.11 MB)

 Non-trainable params: 0 (0.00 B)

## GAN (Basic)

In [6]:

def build_generator():
    model = models.Sequential([
        layers.Dense(256, activation='relu', input_dim=100),
        layers.Dense(32*32*3, activation='tanh'),
        layers.Reshape((32,32,3))
    ])
    return model

def build_discriminator():
    model = models.Sequential([
        layers.Flatten(input_shape=(32,32,3)),
        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

generator = build_generator()
discriminator = build_discriminator()

discriminator.compile(optimizer='adam', loss='binary_crossentropy')

z = layers.Input(shape=(100,))
img = generator(z)

discriminator.trainable = False
validity = discriminator(img)

gan = models.Model(z, validity)
gan.compile(optimizer='adam', loss='binary_crossentropy')


/mnt/d/SRM/Semester 2/DL/project_submission/myenv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/mnt/d/SRM/Semester 2/DL/project_submission/myenv/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
# ================================
# 🔍 OUTPUT VISUALIZATION CELL
# ================================

import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import os

# 🔹 Default image path (change if needed)
default_path = "sample.png"

# 🔹 User input
# user_path = input("Enter image path (or press Enter to use default): ").strip()

# 🔹 Select path
image_path = default_path

# 🔹 Check file
if not os.path.exists(image_path):
    raise FileNotFoundError(f"Image not found: {image_path}")

# 🔹 Load image
img = Image.open(image_path).convert("RGB")
img_resized = img.resize((32, 32))
img_array = np.array(img_resized) / 255.0
img_array = img_array.reshape(1, 32, 32, 3)

# ================================
# 🔹 MODEL OUTPUTS
# ================================

# Denoising
denoised = autoencoder.predict(img_array)

# VAE Reconstruction
vae_output = vae.predict(img_array)

# GAN (random generation)
noise = np.random.normal(size=(1, 100))
gan_output = generator.predict(noise)

# Dummy Caption (replace with model later)
caption = "Enhanced image with AI processing"

# ================================
# 🔹 DISPLAY RESULTS
# ================================

def show_image(title, image):
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")

plt.figure(figsize=(12, 8))

# Original
plt.subplot(2, 2, 1)
show_image("Original Image", img)

# Denoised
plt.subplot(2, 2, 2)
show_image("Denoised Image", denoised[0])

# VAE Output
plt.subplot(2, 2, 3)
show_image("VAE Reconstructed", vae_output[0])

# GAN Output
plt.subplot(2, 2, 4)
show_image("GAN Generated", (gan_output[0] + 1) / 2)  # Normalize tanh output

plt.show()

# ================================
# 🔹 CAPTION OUTPUT
# ================================

print("\n📝 Generated Caption:")
print(caption)